# Inventory Planning & Replenishment Control

One item at one stocking location. Quantities are pieces and time is calendar days. The supplied illustrative demand history is retained. Inventory and order events are generated by the stated policy. No real company or supplier provenance is established.

Run Setup and Baseline first. Each numbered task is independent. Run its BEFORE value, change only the arrow-marked parameter, and rerun the cell. All unchanged settings reset inside the helper. Both comparison runs use the same 60 demand observations and opening stock.

A suggested 30–40 minutes for the activities is an estimate.

## Setup and dataset

One row of `demand_history.csv` or `demand_replay.csv` is one calendar day for SKU-INV-001. `Demand` is requested pieces, including requests that may remain unfilled. January–30 March provides the 90-day estimate. The later 60 days provide a fixed replay and never enter estimation. This is an illustrative scenario comparison, not forecast validation.

In [1]:
from pathlib import Path
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

HELPER_SOURCE = '"""Reproducible, finite-horizon inventory examples. Units: pieces, calendar days."""\nimport math\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom IPython.display import display\n\n\ndef estimate(history):\n    """Fit only the 90-day history, never the 60-day replay."""\n    return float(history.Demand.mean()), float(history.Demand.std(ddof=1))\n\n\ndef position_example(on_order=0):\n    # A separate decision snapshot, not a row from the replay.\n    on_hand, backlog, target = 80, 20, 240\n    ip = on_hand + on_order - backlog\n    return pd.DataFrame([{\'On hand (units)\':on_hand,\'On order (units)\':on_order,\n        \'Backlog (units)\':backlog,\'Inventory position (units)\':ip,\n        \'Target S (units)\':target,\'Order now (units)\':max(0,target-ip)}])\n\n\ndef policy_levels(history, lead_days=4, review_days=7, z=1.65):\n    mu,sigma=estimate(history)\n    ss_lead=z*sigma*math.sqrt(lead_days)\n    ss_protection=z*sigma*math.sqrt(lead_days+review_days)\n    return {\'Mean demand units/day\':mu,\'Demand std units/day\':sigma,\n        \'Lead days\':lead_days,\'Review days\':review_days,\'z\':z,\n        \'Lead-time safety stock units\':ss_lead,\n        \'Continuous ROP units\':math.ceil(mu*lead_days+ss_lead),\n        \'Protection days\':lead_days+review_days,\n        \'Periodic safety stock units\':ss_protection,\n        \'Order-up-to S units\':math.ceil(mu*(lead_days+review_days)+ss_protection)}\n\n\ndef simulate(history, replay, lead_days=4, review_days=7, z=1.65, pack_size=1):\n    """Morning receipt, old-backlog service, review/order, new demand, end snapshot.\n\n    Review on replay days 1, 1+R, ... . An order placed morning t arrives\n    morning t+L. Demand on t occurs after ordering. Backorders persist, with\n    no cancellations/lost sales. Initial stock=220, initial backlog/pipeline=0.\n    Supply is unlimited, full, non-defective, fixed lead time. No forecast updates.\n    """\n    for x in (lead_days,review_days,pack_size):\n        if not isinstance(x,(int,np.integer)) or x<1:raise ValueError(\'Days and pack size must be positive integers\')\n    if not np.isfinite(z) or z<0:raise ValueError(\'z must be finite and nonnegative\')\n    levels=policy_levels(history,lead_days,review_days,z);S=levels[\'Order-up-to S units\']\n    hand,backlog=220,0;pipeline=[];records=[];orders=[]\n    for t,(_,row) in enumerate(replay.iterrows()):\n        opening,old_backlog=hand,backlog\n        received=sum(q for due,q in pipeline if due==t)\n        pipeline=[(due,q) for due,q in pipeline if due!=t]\n        hand+=received\n        cleared=min(hand,backlog);hand-=cleared;backlog-=cleared\n        ip_before=hand+sum(q for _,q in pipeline)-backlog\n        review=t%review_days==0;qty=0\n        if review:\n            qty=math.ceil(max(0,S-ip_before)/pack_size)*pack_size\n            if qty:\n                pipeline.append((t+lead_days,qty));orders.append({\'Order_Date\':row.Date,\'Receipt_Date\':row.Date+pd.Timedelta(days=lead_days),\'Order_Qty\':qty,\'Lead_Time_Days\':lead_days})\n        demand=int(row.Demand);served=min(hand,demand);new_unfilled=demand-served\n        hand-=served;backlog+=new_unfilled\n        records.append({\'Date\':row.Date,\'SKU\':row.SKU,\'Demand\':demand,\n            \'Opening_On_Hand\':opening,\'Opening_Backorder\':old_backlog,\n            \'Receipt_Qty\':received,\'Backlog_Shipped_Qty\':cleared,\n            \'Immediate_Served_Qty\':served,\'New_Unfilled_Qty\':new_unfilled,\n            \'Shipped_Qty\':cleared+served,\'On_Hand_Inventory\':hand,\n            \'Backorder_Qty\':backlog,\'On_Order_Qty\':sum(q for _,q in pipeline),\n            \'Inventory_Position\':hand+sum(q for _,q in pipeline)-backlog,\n            \'Review_Day\':review,\'Order_Qty\':qty,\'Lead_Time_Days\':lead_days,\n            \'Review_Days\':review_days,\'Target_S\':S})\n    d=pd.DataFrame(records);od=pd.DataFrame(orders,columns=[\'Order_Date\',\'Receipt_Date\',\'Order_Qty\',\'Lead_Time_Days\'])\n    return d,od,levels\n\n\ndef metrics(d,orders):\n    demand=int(d.Demand.sum())\n    return {\'Demand units\':demand,\'Immediate fill rate %\':100*d.Immediate_Served_Qty.sum()/demand,\n        \'Fully served demand days %\':100*d.New_Unfilled_Qty.eq(0).mean(),\n        \'Shortage days\':int(d.New_Unfilled_Qty.gt(0).sum()),\n        \'Average end-of-day stock units\':float(d.On_Hand_Inventory.mean()),\n        \'Orders placed\':len(orders),\'Units ordered\':int(d.Order_Qty.sum()),\n        \'Closing stock units\':int(d.On_Hand_Inventory.iloc[-1]),\n        \'Closing backlog units\':int(d.Backorder_Qty.iloc[-1]),\n        \'Closing pipeline units\':int(d.On_Order_Qty.iloc[-1])}\n\n\ndef task1(on_order=0):\n    out=position_example(on_order);display(out);return out\n\n\ndef task2(history,lead_days=4):\n    out=pd.DataFrame([policy_levels(history,lead_days=lead_days)])\n    display(out.round(2));return out\n\n\ndef policy_task(history,replay,changed,parameter):\n    base={\'z\':0.0} if parameter==\'z\' else {}\n    base_d,base_o,base_l=simulate(history,replay,**base)\n    alt={parameter:changed};d,o,l=simulate(history,replay,**alt)\n    before=metrics(base_d,base_o);after=metrics(d,o)\n    out=pd.DataFrame([before,after],index=[f\'Before: {parameter}={0.0 if parameter=="z" else 7 if parameter=="review_days" else 1}\',f\'Current: {parameter}={changed}\'])\n    display(pd.DataFrame([base_l,l],index=out.index)[[\'Lead days\',\'Review days\',\'z\',\'Protection days\',\'Periodic safety stock units\',\'Order-up-to S units\']].round(2))\n    display(out.round(2))\n    fig,axes=plt.subplots(2,1,figsize=(10,6),sharex=True,layout=\'constrained\')\n    axes[0].plot(base_d.Date,base_d.On_Hand_Inventory,label=\'Baseline\',color=\'#5B9BD5\')\n    axes[0].plot(d.Date,d.On_Hand_Inventory,label=\'Current setting\',color=\'#ED7D31\',linestyle=\'--\')\n    axes[0].set(ylabel=\'End stock (units)\',ylim=(0,360));axes[0].legend()\n    axes[1].plot(base_d.Date,base_d.Backorder_Qty,label=\'Baseline backlog\',color=\'#5B9BD5\')\n    axes[1].plot(d.Date,d.Backorder_Qty,label=\'Current backlog\',color=\'#ED7D31\',linestyle=\'--\')\n    axes[1].set(ylabel=\'Backlog (units)\',xlabel=\'Replay date\',ylim=(0,70));fig.autofmt_xdate();plt.show()\n    return d,o,out\n\n\ndef validate(d,orders):\n    """Independent per-row accounting contracts; no KPI output is trusted alone."""\n    assert d.Date.is_monotonic_increasing and not d.duplicated([\'Date\',\'SKU\']).any()\n    assert d.select_dtypes(\'number\').ge(0).all().all()\n    assert (d.Opening_On_Hand+d.Receipt_Qty-d.Shipped_Qty==d.On_Hand_Inventory).all()\n    assert (d.Opening_Backorder+d.New_Unfilled_Qty-d.Backlog_Shipped_Qty==d.Backorder_Qty).all()\n    assert (d.Immediate_Served_Qty+d.New_Unfilled_Qty==d.Demand).all()\n    assert (d.On_Hand_Inventory+d.On_Order_Qty-d.Backorder_Qty==d.Inventory_Position).all()\n    assert ((d.On_Hand_Inventory==0)|(d.Backorder_Qty==0)).all()\n    assert (d.Opening_On_Hand.iloc[1:].to_numpy()==d.On_Hand_Inventory.iloc[:-1].to_numpy()).all()\n    assert (d.Opening_Backorder.iloc[1:].to_numpy()==d.Backorder_Qty.iloc[:-1].to_numpy()).all()\n    assert int(d.Receipt_Qty.sum()+d.On_Order_Qty.iloc[-1])==int(d.Order_Qty.sum())\n    assert int(d.On_Hand_Inventory.iloc[-1]-d.Backorder_Qty.iloc[-1])==220+int(d.Receipt_Qty.sum()-d.Demand.sum())\n    for _,o in orders.iterrows():assert (o.Receipt_Date-o.Order_Date).days==int(o.Lead_Time_Days)\n    for _,r in d.iterrows():\n        exp=orders.loc[orders.Receipt_Date.eq(r.Date),\'Order_Qty\'].sum()\n        assert r.Receipt_Qty==exp\n    return True\n'
exec(compile(HELPER_SOURCE, 'inventory_helpers.py', 'exec'))
HISTORY_CSV = 'Date,SKU,Demand\n2024-01-01,SKU-INV-001,23\n2024-01-02,SKU-INV-001,15\n2024-01-03,SKU-INV-001,21\n2024-01-04,SKU-INV-001,25\n2024-01-05,SKU-INV-001,15\n2024-01-06,SKU-INV-001,17\n2024-01-07,SKU-INV-001,19\n2024-01-08,SKU-INV-001,21\n2024-01-09,SKU-INV-001,17\n2024-01-10,SKU-INV-001,19\n2024-01-11,SKU-INV-001,16\n2024-01-12,SKU-INV-001,21\n2024-01-13,SKU-INV-001,21\n2024-01-14,SKU-INV-001,17\n2024-01-15,SKU-INV-001,22\n2024-01-16,SKU-INV-001,14\n2024-01-17,SKU-INV-001,17\n2024-01-18,SKU-INV-001,17\n2024-01-19,SKU-INV-001,21\n2024-01-20,SKU-INV-001,21\n2024-01-21,SKU-INV-001,13\n2024-01-22,SKU-INV-001,18\n2024-01-23,SKU-INV-001,25\n2024-01-24,SKU-INV-001,17\n2024-01-25,SKU-INV-001,24\n2024-01-26,SKU-INV-001,23\n2024-01-27,SKU-INV-001,24\n2024-01-28,SKU-INV-001,18\n2024-01-29,SKU-INV-001,26\n2024-01-30,SKU-INV-001,18\n2024-01-31,SKU-INV-001,17\n2024-02-01,SKU-INV-001,23\n2024-02-02,SKU-INV-001,26\n2024-02-03,SKU-INV-001,14\n2024-02-04,SKU-INV-001,24\n2024-02-05,SKU-INV-001,24\n2024-02-06,SKU-INV-001,20\n2024-02-07,SKU-INV-001,17\n2024-02-08,SKU-INV-001,27\n2024-02-09,SKU-INV-001,19\n2024-02-10,SKU-INV-001,16\n2024-02-11,SKU-INV-001,17\n2024-02-12,SKU-INV-001,28\n2024-02-13,SKU-INV-001,24\n2024-02-14,SKU-INV-001,27\n2024-02-15,SKU-INV-001,17\n2024-02-16,SKU-INV-001,16\n2024-02-17,SKU-INV-001,19\n2024-02-18,SKU-INV-001,14\n2024-02-19,SKU-INV-001,29\n2024-02-20,SKU-INV-001,20\n2024-02-21,SKU-INV-001,32\n2024-02-22,SKU-INV-001,20\n2024-02-23,SKU-INV-001,17\n2024-02-24,SKU-INV-001,21\n2024-02-25,SKU-INV-001,10\n2024-02-26,SKU-INV-001,27\n2024-02-27,SKU-INV-001,14\n2024-02-28,SKU-INV-001,22\n2024-02-29,SKU-INV-001,16\n2024-03-01,SKU-INV-001,18\n2024-03-02,SKU-INV-001,22\n2024-03-03,SKU-INV-001,13\n2024-03-04,SKU-INV-001,17\n2024-03-05,SKU-INV-001,22\n2024-03-06,SKU-INV-001,20\n2024-03-07,SKU-INV-001,22\n2024-03-08,SKU-INV-001,23\n2024-03-09,SKU-INV-001,29\n2024-03-10,SKU-INV-001,18\n2024-03-11,SKU-INV-001,17\n2024-03-12,SKU-INV-001,25\n2024-03-13,SKU-INV-001,20\n2024-03-14,SKU-INV-001,27\n2024-03-15,SKU-INV-001,18\n2024-03-16,SKU-INV-001,26\n2024-03-17,SKU-INV-001,22\n2024-03-18,SKU-INV-001,21\n2024-03-19,SKU-INV-001,13\n2024-03-20,SKU-INV-001,21\n2024-03-21,SKU-INV-001,22\n2024-03-22,SKU-INV-001,23\n2024-03-23,SKU-INV-001,18\n2024-03-24,SKU-INV-001,22\n2024-03-25,SKU-INV-001,22\n2024-03-26,SKU-INV-001,13\n2024-03-27,SKU-INV-001,17\n2024-03-28,SKU-INV-001,27\n2024-03-29,SKU-INV-001,24\n2024-03-30,SKU-INV-001,21\n'
REPLAY_CSV = 'Date,SKU,Demand\n2024-03-31,SKU-INV-001,15\n2024-04-01,SKU-INV-001,17\n2024-04-02,SKU-INV-001,22\n2024-04-03,SKU-INV-001,27\n2024-04-04,SKU-INV-001,17\n2024-04-05,SKU-INV-001,25\n2024-04-06,SKU-INV-001,15\n2024-04-07,SKU-INV-001,29\n2024-04-08,SKU-INV-001,21\n2024-04-09,SKU-INV-001,14\n2024-04-10,SKU-INV-001,26\n2024-04-11,SKU-INV-001,23\n2024-04-12,SKU-INV-001,18\n2024-04-13,SKU-INV-001,20\n2024-04-14,SKU-INV-001,24\n2024-04-15,SKU-INV-001,23\n2024-04-16,SKU-INV-001,26\n2024-04-17,SKU-INV-001,18\n2024-04-18,SKU-INV-001,21\n2024-04-19,SKU-INV-001,19\n2024-04-20,SKU-INV-001,17\n2024-04-21,SKU-INV-001,7\n2024-04-22,SKU-INV-001,25\n2024-04-23,SKU-INV-001,14\n2024-04-24,SKU-INV-001,24\n2024-04-25,SKU-INV-001,22\n2024-04-26,SKU-INV-001,20\n2024-04-27,SKU-INV-001,20\n2024-04-28,SKU-INV-001,24\n2024-04-29,SKU-INV-001,19\n2024-04-30,SKU-INV-001,25\n2024-05-01,SKU-INV-001,19\n2024-05-02,SKU-INV-001,15\n2024-05-03,SKU-INV-001,21\n2024-05-04,SKU-INV-001,22\n2024-05-05,SKU-INV-001,30\n2024-05-06,SKU-INV-001,21\n2024-05-07,SKU-INV-001,19\n2024-05-08,SKU-INV-001,18\n2024-05-09,SKU-INV-001,10\n2024-05-10,SKU-INV-001,25\n2024-05-11,SKU-INV-001,20\n2024-05-12,SKU-INV-001,20\n2024-05-13,SKU-INV-001,15\n2024-05-14,SKU-INV-001,19\n2024-05-15,SKU-INV-001,22\n2024-05-16,SKU-INV-001,18\n2024-05-17,SKU-INV-001,20\n2024-05-18,SKU-INV-001,22\n2024-05-19,SKU-INV-001,12\n2024-05-20,SKU-INV-001,29\n2024-05-21,SKU-INV-001,18\n2024-05-22,SKU-INV-001,19\n2024-05-23,SKU-INV-001,29\n2024-05-24,SKU-INV-001,15\n2024-05-25,SKU-INV-001,13\n2024-05-26,SKU-INV-001,13\n2024-05-27,SKU-INV-001,12\n2024-05-28,SKU-INV-001,25\n2024-05-29,SKU-INV-001,25\n'

history_path = Path('data/demand_history.csv')
replay_path = Path('data/demand_replay.csv')
history = pd.read_csv(history_path if history_path.exists() else StringIO(HISTORY_CSV), parse_dates=['Date'])
replay = pd.read_csv(replay_path if replay_path.exists() else StringIO(REPLAY_CSV), parse_dates=['Date'])
assert len(history)==90 and len(replay)==60
assert history.Date.max() < replay.Date.min()
assert history.Date.is_unique and replay.Date.is_unique
assert history.Demand.ge(0).all() and replay.Demand.ge(0).all()
print('History:', history.Date.min().date(), 'to', history.Date.max().date(), '| 90 days')
print('Replay:', replay.Date.min().date(), 'to', replay.Date.max().date(), '| 60 days')
display(history.head(3))
print('Historical mean and sample standard deviation (units/day):', tuple(round(x,2) for x in estimate(history)))


History: 2024-01-01 to 2024-03-30 | 90 days
Replay: 2024-03-31 to 2024-05-29 | 60 days


,Date,SKU,Demand
0,2024-01-01,SKU-INV-001,23
1,2024-01-02,SKU-INV-001,15
2,2024-01-03,SKU-INV-001,21


Historical mean and sample standard deviation (units/day): (20.28, 4.36)


## Baseline policy and daily records

The periodic policy reviews every 7 days, with fixed 4-day lead time, z=1.65 and 1-unit packs. Every run opens with 220 pieces, no backlog and no pipeline. Morning receipts first serve old backlog. The planner then reviews/orders if due, and the new day’s demand arrives. An order placed on day t arrives at the start of day t+L.

Demand across days is treated as independent and stable for the normal safety-stock approximation. Real seasonality, forecast error, supplier delays, defects and capacity limits need additional modelling.

`Immediate_Served_Qty` and `New_Unfilled_Qty` are today’s demand flows. `Backorder_Qty` is the outstanding end-of-day state. `Shipped_Qty` includes both old backlog and today’s served demand. `On_Order_Qty` is the remaining end-of-day pipeline. A positive end stock day is a different measure from a day whose demand was fully served.

In [2]:
baseline, baseline_orders, baseline_levels = simulate(history, replay)
validate(baseline, baseline_orders)
display(pd.Series(baseline_levels).round(2))
display(baseline.head(3))
display(pd.Series(metrics(baseline, baseline_orders)).round(2))


inventory_helpers.py:60: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  pipeline.append((t+lead_days,qty));orders.append({'Order_Date':row.Date,'Receipt_Date':row.Date+pd.Timedelta(days=lead_days),'Order_Qty':qty,'Lead_Time_Days':lead_days})


Mean demand units/day            20.28
Demand std units/day              4.36
Lead days                         4.00
Review days                       7.00
z                                 1.65
Lead-time safety stock units     14.38
Continuous ROP units             96.00
Protection days                  11.00
Periodic safety stock units      23.85
Order-up-to S units             247.00
dtype: float64

,Date,SKU,Demand,Opening_On_Hand,Opening_Backorder,Receipt_Qty,Backlog_Shipped_Qty,Immediate_Served_Qty,New_Unfilled_Qty,Shipped_Qty,On_Hand_Inventory,Backorder_Qty,On_Order_Qty,Inventory_Position,Review_Day,Order_Qty,Lead_Time_Days,Review_Days,Target_S
0,2024-03-31,SKU-INV-001,15,220,0,0,0,15,0,15,205,0,27,232,True,27,4,7,247
1,2024-04-01,SKU-INV-001,17,205,0,0,0,17,0,17,188,0,27,215,False,0,4,7,247
2,2024-04-02,SKU-INV-001,22,188,0,0,0,22,0,22,166,0,27,193,False,0,4,7,247


Demand units                      1203.00
Immediate fill rate %              100.00
Fully served demand days %         100.00
Shortage days                        0.00
Average end-of-day stock units      91.22
Orders placed                        9.00
Units ordered                     1155.00
Closing stock units                 37.00
Closing backlog units                0.00
Closing pipeline units             135.00
dtype: float64

## Measures and policy formulas

- Inventory position = on hand + on order − outstanding backlog, all at the same decision time.
- Immediate fill rate = immediately served units ÷ requested units.
- Fully served demand days = days with no newly unfilled units ÷ all replay days.
- Shortage days count newly unfilled demand. Zero end stock alone does not prove shortage.
- Average stock is the arithmetic mean of the 60 end-of-day on-hand snapshots. It is not a continuous time average.
- With historical mean μ and sample standard deviation σ, a fixed protection period T uses approximate buffer z × σ × √T.
- Continuous review reference: ROP = ceiling(μL + zσ√L). It triggers a fixed-Q policy and is not the periodic target.
- Periodic review: S = ceiling(μ(L+R) + zσ√(L+R)); order max(0,S−IP) on review days, rounded up to the pack multiple.

z=1.65 corresponds to about a 95% no-shortage probability over the protected demand interval under the normal approximation. It is neither a guaranteed outcome nor a 95% fill-rate setting.


## Task 1 — Count the pipeline

**BEFORE:** Set `ON_ORDER_UNITS = 0` and run.

**CHANGE:** Set `ON_ORDER_UNITS = 120` and rerun.

**AFTER:** Compare the displayed quantities, measures and plots.

This is a separate morning decision snapshot before allocating the 80 on-hand units to the 20-unit backlog. Target S=240 is given. It is not a replay record.

**THINK**

1. What happens to inventory position and the order quantity when 120 units are already on order?
2. Why can the new order shrink even though physical on-hand stock stays at 80 units?
3. Explain what receipt information the planner must check before relying on the pipeline.

In [3]:
ON_ORDER_UNITS = 0  # ← CHANGE 0 to 120
result_1 = task1(ON_ORDER_UNITS)


,On hand (units),On order (units),Backlog (units),Inventory position (units),Target S (units),Order now (units)
0,80,0,20,60,240,180


## Task 2 — Allow for delivery time

**BEFORE:** Set `LEAD_DAYS = 4` and run.

**CHANGE:** Set `LEAD_DAYS = 7` and rerun.

**AFTER:** Compare the displayed quantities, measures and plots.

Hold review interval at 7 days and z=1.65. This task recalculates settings only. The continuous-review ROP is a reference calculation, separate from the periodic replay.

**THINK**

1. How do the continuous reorder point and periodic order-up-to target change when lead time rises from 4 to 7 days?
2. Why does the periodic policy protect 14 days in the changed run?
3. Explain what these recalculated settings show and what they cannot establish about actual service.

In [4]:
LEAD_DAYS = 4  # ← CHANGE 4 to 7
result_2 = task2(history, LEAD_DAYS)


,Mean demand units/day,Demand std units/day,Lead days,Review days,z,Lead-time safety stock units,Continuous ROP units,Protection days,Periodic safety stock units,Order-up-to S units
0,20.28,4.36,4,7,1.65,14.38,96,11,23.85,247


## Task 3 — Add an uncertainty buffer

**BEFORE:** Set `Z = 0.0` and run.

**CHANGE:** Set `Z = 1.65` and rerun.

**AFTER:** Compare the displayed quantities, measures and plots.

Reset to L=4, R=7 and pack=1. Compare z=0 with z=1.65 on identical replay demand and initial stock. The helper displays both runs.

**THINK**

1. Compare immediate fill rate, shortage days and average stock in the two runs.
2. What does the extra stock protect against in this replay?
3. A manager calls the 100% fill rate a guarantee. How would you explain the evidence?

In [5]:
Z = 0.0  # ← CHANGE 0.0 to 1.65
result_3 = policy_task(history, replay, Z, 'z')


inventory_helpers.py:60: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  pipeline.append((t+lead_days,qty));orders.append({'Order_Date':row.Date,'Receipt_Date':row.Date+pd.Timedelta(days=lead_days),'Order_Qty':qty,'Lead_Time_Days':lead_days})
inventory_helpers.py:60: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  pipeline.append((t+lead_days,qty));orders.append({'Order_Date':row.Date,'Receipt_Date':row.Date+pd.Timedelta(days=lead_days),'Order_Qty':qty,'Lead_Time_Days':lead_days})


,Lead days,Review days,z,Protection days,Periodic safety stock units,Order-up-to S units
Before: z=0.0,4,7,0.0,11,0.0,224
Current: z=0.0,4,7,0.0,11,0.0,224


,Demand units,Immediate fill rate %,Fully served demand days %,Shortage days,Average end-of-day stock units,Orders placed,Units ordered,Closing stock units,Closing backlog units,Closing pipeline units
Before: z=0.0,1203,97.42,95.0,3,70.27,9,1132,14,0,135
Current: z=0.0,1203,97.42,95.0,3,70.27,9,1132,14,0,135


inventory_helpers.py:111: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  axes[1].set(ylabel='Backlog (units)',xlabel='Replay date',ylim=(0,70));fig.autofmt_xdate();plt.show()


## Task 4 — Review more frequently

**BEFORE:** Set `REVIEW_DAYS = 7` and run.

**CHANGE:** Set `REVIEW_DAYS = 3` and rerun.

**AFTER:** Compare the displayed quantities, measures and plots.

Reset to L=4, z=1.65 and pack=1. Compare R=7 with R=3. Each run starts with 220 units and no pipeline/backlog.

**THINK**

1. What happens to the target, average stock, order count and immediate fill rate?
2. Why can more frequent reviews reduce stock while increasing the number of orders?
3. What information is still needed before choosing the shorter review interval?

In [6]:
REVIEW_DAYS = 7  # ← CHANGE 7 to 3
result_4 = policy_task(history, replay, REVIEW_DAYS, 'review_days')


inventory_helpers.py:60: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  pipeline.append((t+lead_days,qty));orders.append({'Order_Date':row.Date,'Receipt_Date':row.Date+pd.Timedelta(days=lead_days),'Order_Qty':qty,'Lead_Time_Days':lead_days})
inventory_helpers.py:60: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  pipeline.append((t+lead_days,qty));orders.append({'Order_Date':row.Date,'Receipt_Date':row.Date+pd.Timedelta(days=lead_days),'Order_Qty':qty,'Lead_Time_Days':lead_days})


,Lead days,Review days,z,Protection days,Periodic safety stock units,Order-up-to S units
Before: review_days=7,4,7,1.65,11,23.85,247
Current: review_days=7,4,7,1.65,11,23.85,247


,Demand units,Immediate fill rate %,Fully served demand days %,Shortage days,Average end-of-day stock units,Orders placed,Units ordered,Closing stock units,Closing backlog units,Closing pipeline units
Before: review_days=7,1203,100.0,100.0,0,91.22,9,1155,37,0,135
Current: review_days=7,1203,100.0,100.0,0,91.22,9,1155,37,0,135


inventory_helpers.py:111: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  axes[1].set(ylabel='Backlog (units)',xlabel='Replay date',ylim=(0,70));fig.autofmt_xdate();plt.show()


## Task 5 — Order in pack multiples

**BEFORE:** Set `PACK_SIZE = 1` and run.

**CHANGE:** Set `PACK_SIZE = 50` and rerun.

**AFTER:** Compare the displayed quantities, measures and plots.

Reset to L=4, R=7 and z=1.65. Round every positive order up to the pack multiple. A zero gap still gives a zero order.

**THINK**

1. What happens to average stock, ordered units, order count and immediate fill rate?
2. Why does pack rounding add stock even though target S stays unchanged?
3. Explain whether the results justify the 50-unit pack rule.

In [7]:
PACK_SIZE = 1  # ← CHANGE 1 to 50
result_5 = policy_task(history, replay, PACK_SIZE, 'pack_size')


inventory_helpers.py:60: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  pipeline.append((t+lead_days,qty));orders.append({'Order_Date':row.Date,'Receipt_Date':row.Date+pd.Timedelta(days=lead_days),'Order_Qty':qty,'Lead_Time_Days':lead_days})
inventory_helpers.py:60: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  pipeline.append((t+lead_days,qty));orders.append({'Order_Date':row.Date,'Receipt_Date':row.Date+pd.Timedelta(days=lead_days),'Order_Qty':qty,'Lead_Time_Days':lead_days})


,Lead days,Review days,z,Protection days,Periodic safety stock units,Order-up-to S units
Before: pack_size=1,4,7,1.65,11,23.85,247
Current: pack_size=1,4,7,1.65,11,23.85,247


,Demand units,Immediate fill rate %,Fully served demand days %,Shortage days,Average end-of-day stock units,Orders placed,Units ordered,Closing stock units,Closing backlog units,Closing pipeline units
Before: pack_size=1,1203,100.0,100.0,0,91.22,9,1155,37,0,135
Current: pack_size=1,1203,100.0,100.0,0,91.22,9,1155,37,0,135


inventory_helpers.py:111: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  axes[1].set(ylabel='Backlog (units)',xlabel='Replay date',ylim=(0,70));fig.autofmt_xdate();plt.show()


## Planning interpretation

Explain how pipeline commitments, lead time, uncertainty, review frequency and pack constraints affect the order decision. Use the displayed output and state one condition that could change your recommendation.

The replay uses unlimited, complete, on-time supply and one demand path. It has no warm-up or repeated random trials. Orders after the last review can remain outstanding beyond 29 May. The tables therefore show closing stock, backlog and pipeline. These runs do not establish an optimal policy or a full cost comparison.